## NHIES Analysis
### Point 3 - National/Regional g/c/d estimates of fortification vectors
### 17 November 2020

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 210)
pd.set_option("display.max_columns", None)

## National-level data

In [3]:
# Read in ADePT macro results (Table 3.1)
fn = 'table_3_1_2020.csv'
path = Path.cwd().joinpath('data', fn)
total_df = pd.read_csv(path, skiprows=1)

## Functions to Process/Clean data and Analyse

In [4]:
# Tidy columns
def tidy_columns(df):
    # Rename first column
    df = df.rename(columns={'Unnamed: 0': 'Food_Item', 'Average edible quantity (g/capita/day)': 'avg_edible_qty_gcd'})

    # Clean whitespace from Food_Item
    df.loc[:, 'Food_Item'] = df.Food_Item.str.strip()

    # Create separate values for Food_Item column
    df['COICOP'] = [x.split(" ", 1)[0][:] for x in df.Food_Item]
    df['Label'] = [x.split(" ", 1)[1][:] for x in df.Food_Item]

    # Strip whitespace
    df.loc[:, 'COICOP'] = [x.strip() for x in df['COICOP']]
    df.loc[:, 'Label'] = [x.strip() for x in df['Label']]

    df = df.drop(['Average quantity as purchased (g/capita/day)', 'Average food consumption in monetary value (LCU/capita/day)',
    'Average dietary energy consumption (kcal/capita/day)', 'Median dietary energy unit value (LCU/1000 kcal)'], axis=1)

    col_order = ['Food_Item', 'COICOP', 'Label', 'avg_edible_qty_gcd']

    df = df[col_order]

    return df

In [5]:
# Classification of fortification vehicles
wheat_items = ['Bread (white, brown, whole wheat, rye, maize, etc.)', 'Brotchen', 'Biscuits, rusks', 'Vetkoek', 'Macaroni, spaghetti, noodles', 'Cakes (all types)', 'Pies & pizzas', 'Bread/ cake flour (all types)']

maize_items = ['Traditional bread, ash bread, oshikwiila, oshima, omungome', 'Maize meal/ grain/ samp']

mahangu_items = ['Mahangu meal/ grain/ samp', 'Mageu/Oshikundu']

salt_items = ['Bread (white, brown, whole wheat, rye, maize, etc.)', 'Brotchen', 'Biscuits, rusks', 'Vetkoek', 'Traditional bread, ash bread, oshikwiila, oshima, omungome',
'Macaroni, spaghetti, noodles', 'Cakes (all types)', 'Pies & pizzas', 'Bacon', 'Ham', 'Biltong', 'Smoked meat, other than bacon & ham', 'Polony', 'Sausages, other than polony', 'Canned meat', 'Smoked fish', 'Dried fish', 'Bottled/ Tinned fish', 'Cheese, all types', 'Chips & crisps', 'Tomato Sauces', 'Salt', 'Spices & condiments', 'Ready-made soup', 'Powdered soup']

milk_items = ['Fresh milk', 'Long life milk', 'Fresh milk - Low fat', 'Long life - Low fat', 'Clotted/ Cultured milk']

oil_items = ['Cooking oil', 'Olive oil']

sugar_items = ['Sugar']

In [6]:
# Food items content
# Cereal flour
cereal_flour_dict = {
    'Bread (white, brown, whole wheat, rye, maize, etc.)': 75,
    'Brotchen': 75,
    'Biscuits, rusks': 85,
    'Vetkoek': 75,
    'Macaroni, spaghetti, noodles': 45,
    'Cakes (all types)': 65,
    'Pies & pizzas': 70,
    'Bread/ cake flour (all types)': 100,
    'Traditional bread, ash bread, oshikwiila, oshima, omungome': 75,
    'Maize meal/ grain/ samp': 100,
    'Mahangu meal/ grain/ samp': 100,
    'Mageu/Oshikundu': 100
}

# Salt
salt_dict = {
    'Bread (white, brown, whole wheat, rye, maize, etc.)': 1.6,
    'Brotchen': 1.6,
    'Biscuits, rusks': 0.3,
    'Vetkoek': 0.07,
    'Traditional bread, ash bread, oshikwiila, oshima, omungome': 1.6,
    'Macaroni, spaghetti, noodles': 0.0,
    'Cakes (all types)': 1.05,
    'Pies & pizzas': 0.96,
    'Bacon': 3.99,
    'Ham': 3.3,
    'Biltong': 5.53,
    'Smoked meat, other than bacon & ham': 2.4,
    'Polony': 2.55,
    'Sausages, other than polony': 3.24,
    'Canned meat': 2.5,
    'Smoked fish': 1.7,
    'Dried fish': 13.9,
    'Bottled/ Tinned fish': 0.7,
    'Cheese, all types': 2,
    'Chips & crisps': 2.5,
    'Tomato Sauces': 1.46,
    'Salt': 100,
    'Spices & condiments': 30,
    'Ready-made soup': 0.85,
    'Powdered soup': 13.66
}

In [7]:
def add_fort_content(df):
    # Add cereal content column
    df['cereal_content'] = [cereal_flour_dict[x] if x in cereal_flour_dict.keys() else np.nan for x in df.Label]

    # Add salt content column
    df['salt_content'] = [salt_dict[x] if x in salt_dict.keys() else np.nan for x in df.Label]

    # Add absolute values
    df['cereal_flour_gcd'] = (df.avg_edible_qty_gcd / 100) * df.cereal_content
    df['salt_gcd'] = (df.avg_edible_qty_gcd / 100) * df.salt_content

    return df

In [8]:
def clean_df(df):
    # Drop row if all values missing 
    df = df.dropna(thresh=5)

    # Tidy columns
    df = tidy_columns(df)

    # Add fortification info
    df = add_fort_content(df)

    return df

In [9]:
def generate_output(df, region):
    output = pd.DataFrame({
        'Wheat_gcd_sum': df[df.Label.isin(wheat_items)].cereal_flour_gcd.sum(),
        'Maize_gcd_sum': df[df.Label.isin(maize_items)].cereal_flour_gcd.sum(),
        'Mahangu_gcd_sum': df[df.Label.isin(mahangu_items)].cereal_flour_gcd.sum(),
        'Salt_gcd_sum': df[df.Label.isin(salt_items)].salt_gcd.sum(),
        'Milk_gcd_sum': df[df.Label.isin(milk_items)].avg_edible_qty_gcd.sum(),
        'Oil_gcd_sum': df[df.Label.isin(oil_items)].avg_edible_qty_gcd.sum(),
        'Sugar_gcd_sum': df[df.Label.isin(sugar_items)].avg_edible_qty_gcd.sum()
    }, index=[region])

    return output

## Regional-level data

In [10]:
# Read in ADePT macro results (Table 3.5)
fn = 'table_3_5_2020.csv'
path = Path.cwd().joinpath('data', fn)
df = pd.read_csv(path, skiprows=1)

### Create separate dfs for each region

In [11]:
# Get row index for region values
region_list = ['!Karas', 'Erongo', 'Hardap', 'Kavango East', 'Kavango West', 'Khomas', 'Kunene', 'Ohangwena', 'Omaheke', 'Omusati', 'Oshana', 'Oshikoto', 'Otjozondjupa', 'Zambezi']

region_idxs = df.index[df['Unnamed: 0'].isin(region_list)].tolist()
# Add last value
region_idxs.insert(len(region_idxs), df.shape[0])
print(region_idxs)

[2, 167, 340, 492, 630, 759, 934, 1073, 1222, 1357, 1498, 1663, 1813, 1984, 2112]


In [12]:
# List comp to get row ranges for regional dfs
region_idxs_range_list = [range(region_idxs[i],region_idxs[i+1]-1) for i in range(len(region_idxs)-1)]
# Create dict with region names
region_idxs_dict = dict(zip(region_list, region_idxs_range_list))
print(region_idxs_dict)

{'!Karas': range(2, 166), 'Erongo': range(167, 339), 'Hardap': range(340, 491), 'Kavango East': range(492, 629), 'Kavango West': range(630, 758), 'Khomas': range(759, 933), 'Kunene': range(934, 1072), 'Ohangwena': range(1073, 1221), 'Omaheke': range(1222, 1356), 'Omusati': range(1357, 1497), 'Oshana': range(1498, 1662), 'Oshikoto': range(1663, 1812), 'Otjozondjupa': range(1813, 1983), 'Zambezi': range(1984, 2111)}


In [13]:
# List comp to get regional dfs
list_of_region_dfs = [df.loc[region_idxs_dict[region]] for region in region_list]

In [14]:
# Create dict with region names, dfs
dict_of_region_dfs = dict(zip(region_list, list_of_region_dfs))

In [15]:
# Add national to dictionary
dict_of_region_dfs['Total'] = total_df

## Clean data and generate output

In [16]:
def run_analysis(dict_of_dfs):
    output_df = pd.concat([generate_output(clean_df(v), k) for k,v in dict_of_dfs.items()], axis=0)
    return output_df

In [17]:
output_df = run_analysis(dict_of_region_dfs)

fn = 'output.csv'
path = Path.cwd().joinpath('output',fn)

output_df.to_csv(path)